# 05 — Tests du pipeline (annexe D)

Chaque test vérifie une propriété attendue. Si un test échoue, la cellule s'arrête avec une `AssertionError`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # pour importer le code de src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nutrihfr import config as C

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})
C.DOSSIER_FIGURES.mkdir(parents=True, exist_ok=True)
C.DOSSIER_TABLES.mkdir(parents=True, exist_ok=True)

In [ ]:
from nutrihfr.pipeline import ChargeurINCA3, CalculApports
individus, consommations, apports_anses = ChargeurINCA3().charger()
apports = CalculApports(consommations, individus).par_individu()

**Test 1** — Aucune quantité négative.

In [ ]:
assert (consommations["qte_conso"] >= 0).all()
print("Test 1 OK")

**Test 2** — Tous les nutriments ont une unité connue.

In [ ]:
unites = [unite for (_, unite, _) in C.NUTRIMENTS.values()]
assert all(u in ["kcal", "g", "mg", "µg"] for u in unites)
print("Test 2 OK :", dict(zip(C.NUTRIMENTS, unites)))

**Test 3** — Tout aliment consommé a une composition (sauf exceptions connues, listées).

In [ ]:
sans_compo = consommations[consommations[list(C.NUTRIMENTS)].isna().any(axis=1)]
print("Exceptions (aliments sans composition) :", sans_compo["aliment_libelle_INCA3"].unique())
assert len(sans_compo) <= 10, "nouveaux aliments sans composition !"
print("Test 3 OK")

**Test 4** — Pour un aliment connu, 100 g × composition donne le résultat attendu (exemple : 200 g d'un lait à 120 mg de calcium pour 100 g → 240 mg).

In [ ]:
faux = pd.DataFrame({"NOIND": [1], "R24_nombre": [1], "qte_conso_pond": [200.0],
                     **{n: [0.0] for n in C.NUTRIMENTS}})
faux["calcium"] = 120.0
resultat = CalculApports(faux, None).par_ligne()["calcium"].iloc[0]
assert abs(resultat - 240) < 1e-9
print("Test 4 OK :", resultat, "mg")

**Test 5** — 0 g consommé donne 0 apport.

In [ ]:
faux["qte_conso_pond"] = 0.0
assert (CalculApports(faux, None).par_ligne()[list(C.NUTRIMENTS)] == 0).all().all()
print("Test 5 OK")

**Test 6** — La somme des pondérations est cohérente avec la population adulte (≈ 45-48 millions de 18-79 ans en France métropolitaine).

In [ ]:
total = individus["ponderation"].sum()
assert 40e6 < total < 50e6
print(f"Test 6 OK : {total:,.0f}")

**Test 7** — L'énergie recalculée avec les coefficients d'Atwater est proche de l'énergie indiquée (écart < 10 % pour au moins 95 % des lignes).

In [ ]:
co = consommations[consommations["aet"] > 20]
atwater = 4 * co["proteines"] + 4 * co["glucides"] + 9 * co["lipides"] + 2 * co["fibres"] + 7 * co["alcool"]
part = ((atwater - co["aet"]).abs() / co["aet"] < 0.10).mean()
assert part >= 0.95
print(f"Test 7 OK : {part:.1%} des lignes")

**Test 8 (bonus)** — Nos apports sont identiques à ceux publiés par l'Anses.

In [ ]:
m = apports.merge(apports_anses, on="NOIND", suffixes=("", "_anses"))
for n in C.NUTRIMENTS:
    assert (m[n] - m[n + "_anses"]).abs().max() < 0.01, n
print("Test 8 OK : apports identiques pour les", len(m), "adultes")

✅ Si toutes les cellules affichent « OK », le pipeline est fiable.